# 🍷 SỬ DỤNG APACHE SPARK DỰ ĐOÁN CHẤT LƯỢNG RƯỢU

## Mục tiêu
- Khảo sát và phân tích bộ dữ liệu Wine Quality.
- Tiền xử lý dữ liệu bằng Apache Spark.
- Phân tích mối quan hệ giữa các đặc trưng hóa học và chất lượng rượu.
- Xây dựng các mô hình Machine Learning bằng Spark MLlib.
- So sánh hiệu quả các mô hình.
- Dự đoán chất lượng của một mẫu rượu mới.

## Bài toán
Biến mục tiêu: `quality`

Bài toán được xây dựng dưới dạng phân loại đa lớp với các mức chất lượng rượu từ 4 đến 8.

# 1. CÀI ĐẶT MÔI TRƯỜNG VÀ APACHE SPARK

In [ ]:
!pip install -q pyspark

## 1.1. Import các thư viện cần thiết

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from sklearn.metrics import confusion_matrix, classification_report

import warnings
warnings.filterwarnings("ignore")

## 1.2. Khởi tạo SparkSession

In [ ]:
spark = SparkSession.builder.appName("Wine Quality Prediction").getOrCreate()
print("Spark đã được khởi tạo thành công!")
print("Spark version:", spark.version)

# 2. ĐỌC VÀ KHẢO SÁT DỮ LIỆU

In [ ]:
DATA_PATH = "/content/winequality.csv"

df_spark = spark.read.csv(
    DATA_PATH,
    header=True,
    inferSchema=True
)

print("Đọc dữ liệu thành công!")
print("Số dòng:", df_spark.count())
print("Số cột:", len(df_spark.columns))

## 2.1. Hiển thị dữ liệu

In [ ]:
df_spark.show(10, truncate=False)

## 2.2. Kiểm tra cấu trúc dữ liệu

In [ ]:
df_spark.printSchema()

## 2.3. Danh sách các thuộc tính

In [ ]:
for i, c in enumerate(df_spark.columns, 1):
    print(f"{i}. {c}")

## 2.4. Kích thước và thống kê mô tả

In [ ]:
print("Số dòng:", df_spark.count())
print("Số cột:", len(df_spark.columns))
df_spark.describe().show()

# 3. KIỂM TRA VÀ XỬ LÝ DỮ LIỆU

## 3.1. Kiểm tra giá trị thiếu

In [ ]:
null_counts = df_spark.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_spark.columns
])
null_counts.show()

## 3.2. Tổng số giá trị thiếu

In [ ]:
null_row = null_counts.collect()[0]
total_null = sum(value if value is not None else 0 for value in null_row)
print("Tổng số giá trị thiếu:", total_null)

## 3.3. Xử lý giá trị thiếu

In [ ]:
df_clean = df_spark.dropna()
print("Số dòng ban đầu:", df_spark.count())
print("Số dòng sau khi xử lý:", df_clean.count())
print("Số dòng bị loại:", df_spark.count() - df_clean.count())

## 3.4. Kiểm tra các giá trị bất hợp lệ

In [ ]:
invalid_count = df_clean.filter(
    (col("pH") <= 0) | (col("pH") > 14) |
    (col("density") <= 0) | (col("alcohol") <= 0)
).count()
print("Số dòng có giá trị bất hợp lệ:", invalid_count)

## 3.5. Xử lý các giá trị bất hợp lệ

In [ ]:
df_clean = df_clean.filter(
    (col("pH") > 0) & (col("pH") <= 14) &
    (col("density") > 0) & (col("alcohol") > 0)
)
print("Số dòng sau khi xử lý:", df_clean.count())

## 3.6. Phân tích Outlier bằng IQR

In [ ]:
numeric_cols = [
    "fixed acidity", "volatile acidity", "citric acid",
    "residual sugar", "chlorides", "free sulfur dioxide",
    "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"
]

for c in numeric_cols:
    q1, q3 = df_clean.approxQuantile(c, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count_outlier = df_clean.filter((col(c) < lower) | (col(c) > upper)).count()
    print(f"{c}: Q1={q1:.3f}, Q3={q3:.3f}, Outlier={count_outlier}")

# 4. XỬ LÝ BIẾN MỤC TIÊU QUALITY

In [ ]:
df_clean = df_clean.filter(col("quality").isin([4, 5, 6, 7, 8]))

df_clean = df_clean.withColumn(
    "label",
    when(col("quality") == 4, 0)
    .when(col("quality") == 5, 1)
    .when(col("quality") == 6, 2)
    .when(col("quality") == 7, 3)
    .when(col("quality") == 8, 4)
)

df_clean.groupBy("quality", "label").count().orderBy("quality").show()

# 5. PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA)

In [ ]:
df = df_clean.toPandas()
print("Kích thước dữ liệu:", df.shape)
display(df.head())

## 5.1. Phân bố chất lượng rượu

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="quality")
plt.title("Phân bố chất lượng rượu")
plt.xlabel("Quality")
plt.ylabel("Số lượng mẫu")
plt.show()

## 5.2. Thống kê số lượng theo Quality

In [ ]:
display(df['quality'].value_counts().sort_index().to_frame('Số lượng'))

## 5.3. Thống kê mô tả các thuộc tính

In [ ]:
display(df[numeric_cols].describe().T)

## 5.4. Phân bố các thuộc tính

In [ ]:
selected_plot_cols = ["alcohol", "volatile acidity", "sulphates", "citric acid", "density"]
for c in selected_plot_cols:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df, x=c, kde=True)
    plt.title(f"Phân bố {c}")
    plt.xlabel(c)
    plt.ylabel("Số lượng")
    plt.show()

## 5.5. Ma trận tương quan

In [ ]:
corr = df[numeric_cols + ["quality"]].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Ma trận tương quan giữa các thuộc tính")
plt.show()

## 5.6. Phân tích tương quan với Quality

In [ ]:
corr_quality = corr["quality"].drop("quality").sort_values(key=abs, ascending=False)
display(corr_quality.to_frame("Correlation"))

## 5.7. Biểu đồ tương quan với Quality

In [ ]:
plt.figure(figsize=(8, 6))
corr_quality.sort_values().plot(kind="barh")
plt.title("Mức độ tương quan giữa các thuộc tính và Quality")
plt.xlabel("Correlation")
plt.ylabel("Feature")
plt.show()

## 5.8. Phân tích mối quan hệ giữa các thuộc tính

In [ ]:
pair_cols = ["alcohol", "volatile acidity", "sulphates", "citric acid", "quality"]
sns.pairplot(df[pair_cols], hue="quality")
plt.show()

# 6. CHUẨN BỊ DỮ LIỆU CHO SPARK ML

## 6.1. VectorAssembler

In [ ]:
feature_cols = [
    "fixed acidity", "volatile acidity", "citric acid",
    "residual sugar", "chlorides", "free sulfur dioxide",
    "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_vector = assembler.transform(df_clean)

df_vector.select("quality", "label", "features").show(5, truncate=False)

## 6.2. Chuẩn hóa dữ liệu bằng StandardScaler

In [ ]:
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(df_vector)
df_scaled = scaler_model.transform(df_vector)

df_scaled.select("label", "features", "scaledFeatures").show(5, truncate=False)

## 6.3. Chia dữ liệu Train/Test

In [ ]:
train_df, test_df = df_scaled.randomSplit([0.8, 0.2], seed=42)
print("Số mẫu Train:", train_df.count())
print("Số mẫu Test :", test_df.count())

## 6.4. Kiểm tra phân bố Label

In [ ]:
print("Phân bố nhãn trong tập Train:")
train_df.groupBy("label").count().orderBy("label").show()

print("Phân bố nhãn trong tập Test:")
test_df.groupBy("label").count().orderBy("label").show()

# 7. XÂY DỰNG MÔ HÌNH MACHINE LEARNING

## 7.1. Logistic Regression

In [ ]:
lr = LogisticRegression(featuresCol="scaledFeatures", labelCol="label", maxIter=100)
lr_model = lr.fit(train_df)
lr_pred = lr_model.transform(test_df)
lr_pred.select("label", "prediction", "probability").show(10, truncate=False)

## 7.2. Decision Tree

In [ ]:
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5, seed=42)
dt_model = dt.fit(train_df)
dt_pred = dt_model.transform(test_df)
dt_pred.select("label", "prediction").show(10)

## 7.3. Random Forest

In [ ]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=8,
    seed=42
)
rf_model = rf.fit(train_df)
rf_pred = rf_model.transform(test_df)
rf_pred.select("label", "prediction").show(10)

# 8. ĐÁNH GIÁ MÔ HÌNH

In [ ]:
def evaluate_model(predictions, model_name):
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction"
    )
    accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
    precision = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})
    recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})
    f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})
    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

## 8.1. So sánh hiệu quả các mô hình

In [ ]:
results = [
    evaluate_model(lr_pred, "Logistic Regression"),
    evaluate_model(dt_pred, "Decision Tree"),
    evaluate_model(rf_pred, "Random Forest")
]
results_df = pd.DataFrame(results)
display(results_df)

## 8.2. Biểu đồ so sánh các mô hình

In [ ]:
results_melted = results_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(10, 6))
sns.barplot(data=results_melted, x="Model", y="Score", hue="Metric")
plt.title("So sánh hiệu quả các mô hình")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.show()

## 8.3. Lựa chọn mô hình tốt nhất

In [ ]:
best_model = results_df.loc[results_df["F1"].idxmax()]
print("Mô hình tốt nhất dựa trên F1-score:")
display(best_model.to_frame().T)

## 8.4. Confusion Matrix - Random Forest

In [ ]:
rf_result_pd = rf_pred.select("label", "prediction").toPandas()
cm = confusion_matrix(rf_result_pd["label"], rf_result_pd["prediction"])

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Random Forest")
plt.show()

## 8.5. Classification Report

In [ ]:
print(
    classification_report(
        rf_result_pd["label"],
        rf_result_pd["prediction"],
        target_names=["Quality 4", "Quality 5", "Quality 6", "Quality 7", "Quality 8"]
    )
)

# 9. PHÂN TÍCH FEATURE IMPORTANCE

In [ ]:
importance = rf_model.featureImportances.toArray()

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importance
}).sort_values("Importance", ascending=False)

display(importance_df)

## 9.1. Biểu đồ Feature Importance

In [ ]:
plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Feature Importance - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

# 10. TỐI ƯU MÔ HÌNH BẰNG CROSS VALIDATION

In [ ]:
param_grid = ParamGridBuilder()     .addGrid(rf.numTrees, [50, 100])     .addGrid(rf.maxDepth, [5, 8])     .build()

evaluator_cv = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_cv,
    numFolds=3,
    seed=42
)

cv_model = cv.fit(train_df)
cv_pred = cv_model.transform(test_df)
cv_f1 = evaluator_cv.evaluate(cv_pred)

print("F1-score sau Cross Validation:", round(cv_f1, 4))

# 11. DỰ ĐOÁN CHẤT LƯỢNG MỘT MẪU RƯỢU MỚI

In [ ]:
sample = spark.createDataFrame(
    [(7.4, 0.70, 0.00, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4)],
    feature_cols
)
sample.show()

## 11.1. Vector hóa mẫu mới

In [ ]:
sample_vector = assembler.transform(sample)
sample_vector.select("features").show(truncate=False)

## 11.2. Dự đoán

In [ ]:
sample_prediction = rf_model.transform(sample_vector)
sample_prediction.select("prediction", "probability").show(truncate=False)

## 11.3. Chuyển Label về Quality

In [ ]:
label_to_quality = {0: 4, 1: 5, 2: 6, 3: 7, 4: 8}

prediction = sample_prediction.select("prediction").collect()[0]["prediction"]
quality_prediction = label_to_quality[int(prediction)]

print(f"🍷 Chất lượng rượu dự đoán: {quality_prediction}")

# 12. LƯU MÔ HÌNH

In [ ]:
rf_model.write().overwrite().save("/content/wine_quality_rf_model")
print("Đã lưu mô hình Random Forest bằng Spark.")

# 13. KẾT LUẬN

Qua quá trình thực hiện, nhóm đã:

- Sử dụng Apache Spark để đọc và xử lý bộ dữ liệu Wine Quality.
- Kiểm tra cấu trúc, dữ liệu thiếu và các giá trị bất hợp lệ.
- Phân tích phân bố chất lượng rượu.
- Phân tích mối tương quan giữa các thuộc tính hóa học và Quality.
- Sử dụng VectorAssembler để chuyển đổi dữ liệu thành vector đặc trưng.
- Sử dụng StandardScaler để chuẩn hóa dữ liệu.
- Xây dựng các mô hình Logistic Regression, Decision Tree và Random Forest bằng Spark MLlib.
- Đánh giá mô hình bằng Accuracy, Precision, Recall và F1-score.
- Phân tích Feature Importance của mô hình Random Forest.
- Thực hiện dự đoán chất lượng cho một mẫu rượu mới.

Mô hình có F1-score cao nhất được lựa chọn làm mô hình phù hợp nhất cho bài toán dự đoán chất lượng rượu.

# 14. HƯỚNG PHÁT TRIỂN

- Mở rộng bộ dữ liệu với nhiều mẫu rượu hơn.
- Thử nghiệm thêm các thuật toán Machine Learning.
- Tối ưu tham số mô hình bằng Spark CrossValidator.
- Xây dựng giao diện nhập thông số hóa học của rượu.
- Triển khai mô hình thành ứng dụng dự đoán chất lượng rượu.
- Triển khai Spark trên môi trường cluster để đánh giá khả năng xử lý dữ liệu lớn.